In [ ]:
# 1. Install PostgreSQL and dev packages
!apt-get -qq update > /dev/null
!apt-get -qq install postgresql postgresql-contrib postgresql-server-dev-all -y > /dev/null

# 2. Build and install pgvector
!git clone --quiet https://github.com/pgvector/pgvector.git
!cd pgvector && make clean > /dev/null && make > /dev/null && make install > /dev/null

# 3. Start PostgreSQL and set password
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!pip install -q psycopg2-binary sentence-transformers

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 51.0 MB/s eta 0:00:00


In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer

# 1. Load small local embedder
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Connect to Postgres & create table
conn = psycopg2.connect("dbname=postgres user=postgres password=postgres host=localhost port=5432")
conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("DROP TABLE IF EXISTS faqs;")
cur.execute("""
    CREATE TABLE faqs (
        id SERIAL PRIMARY KEY,
        question TEXT,
        answer TEXT,
        embedding vector(384)
    );
""")

# 3. Insert FAQs
faqs = [
    ("How do I reset my password?", "Go to settings -> Security -> Click 'Reset Password'."),
    ("What are your refund policies?", "You can claim a 100% refund within 14 days of purchase."),
    ("How can I track my order shipping?", "Check your email for the courier tracking link."),
]

for q, a in faqs:
    vec = model.encode(q).tolist()
    cur.execute("INSERT INTO faqs (question, answer, embedding) VALUES (%s, %s, %s::vector)", (q, a, str(vec)))

# 4. User query
user_query = "I cannot log in because I lost my secret code"
query_vec = str(model.encode(user_query).tolist())

# 5. Query using Cosine Distance (<=>)
cur.execute("""
    SELECT question, answer, (1 - (embedding <=> %s::vector)) AS similarity
    FROM faqs
    ORDER BY embedding <=> %s::vector
    LIMIT 1;
""", (query_vec, query_vec))

match = cur.fetchone()
print(f"\nUser Asked: '{user_query}'")
print(f"Matched FAQ: '{match[0]}' (Score: {match[2]:.2f})")
print(f"Direct Answer: {match[1]}")

cur.close()
conn.close()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


User Asked: 'I forgot how to use the product'
Matched FAQ: 'How do I reset my password?' (Score: 0.32)
Direct Answer: Go to settings -> Security -> Click 'Reset Password'.
